<a href="https://colab.research.google.com/github/ChristianAgyapong/Medgemma/blob/main/TASK_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import the tools we need**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

**Load the dataset ( no CSV needed)**

In [ ]:
# Load California housing data (real dataset about houses in California)
housing = fetch_california_housing(as_frame=True)
df = housing.frame   # this gives us a nice pandas DataFrame

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTarget (what we predict):")
print("MedHouseVal = median house value in $100,000 units")

# Quick look
print("\nFirst 3 rows:")
print(df.head(3))

Dataset shape: (20640, 9)

Columns:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']

Target (what we predict):
MedHouseVal = median house value in $100,000 units

First 3 rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  


**Prepare X (features) and y (target)**

In [ ]:
# Features (X) = everything except the target
# Target (y) = house price
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

print("Features shape:", X.shape)
print("Target shape :", y.shape)

# Optional but helpful: scale the features (makes coefficients easier to compare)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)  # keep column names

print("\nAfter scaling (first 3 rows):")
print(X.head(3))

Features shape: (20640, 8)
Target shape : (20640,)

After scaling (first 3 rows):
     MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  2.344766  0.982143  0.628559  -0.153758   -0.974429 -0.049597  1.052548   
1  2.332238 -0.607019  0.327041  -0.263336    0.861439 -0.092512  1.043185   
2  1.782699  1.856182  1.155620  -0.049016   -0.820777 -0.025843  1.038503   

   Longitude  
0  -1.327835  
1  -1.322844  
2  -1.332827  


**Split into train + test**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # 20% for testing
    random_state=42         # same split every time
)

print("Training examples:", X_train.shape[0])
print("Testing examples :", X_test.shape[0])

Training examples: 16512
Testing examples : 4128


**Change words (categories) to numbers**

In [ ]:
# Create and train the model (just 2 lines!)
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully! ✓")

Model trained successfully! ✓


**Scale (make numbers similar size) – Age and Fare only**

In [ ]:
y_pred = model.predict(X_test)

# Compare first few real vs predicted prices
comparison = pd.DataFrame({
    'Real price ($100k)': y_test.values[:8],
    'Predicted price ($100k)': y_pred[:8].round(2)
})
print("Example predictions:")
print(comparison)

Example predictions:
   Real price ($100k)  Predicted price ($100k)
0             0.47700                     0.72
1             0.45800                     1.76
2             5.00001                     2.71
3             2.18600                     2.84
4             2.78000                     2.60
5             1.58700                     2.01
6             1.98200                     2.65
7             1.57500                     2.17


**Separate features (X) and target (y)**

In [ ]:
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)   # Root MSE — same unit as price, easier to understand

print(f"R-squared     : {r2:.3f}  → means the model explains about {r2*100:.1f}% of price differences")
print(f"Mean Squared Error (MSE) : {mse:.3f}")
print(f"Root Mean Squared Error  : {rmse:.3f}  → average prediction error ≈ ${rmse*100_000:,.0f}")

R-squared     : 0.576  → means the model explains about 57.6% of price differences
Mean Squared Error (MSE) : 0.556
Root Mean Squared Error  : 0.746  → average prediction error ≈ $74,558


In [ ]:
# Create nice table of coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_.round(4)
}).sort_values('Coefficient', ascending=False)

print("Intercept (base price when all features = 0):", round(model.intercept_, 3))
print("\nCoefficients (sorted from most positive to most negative):")
print(coef_df)

# Quick explanation print
print("\nInterpretation examples:")
print("• Positive coefficient → when feature ↑, house price ↑")
print("• Negative coefficient → when feature ↑, house price ↓")
print("• Larger |coefficient| → stronger effect on price")

Intercept (base price when all features = 0): 2.068

Coefficients (sorted from most positive to most negative):
      Feature  Coefficient
0      MedInc       0.8524
3   AveBedrms       0.3711
1    HouseAge       0.1224
4  Population      -0.0023
5    AveOccup      -0.0366
2    AveRooms      -0.3051
7   Longitude      -0.8689
6    Latitude      -0.8966

Interpretation examples:
• Positive coefficient → when feature ↑, house price ↑
• Negative coefficient → when feature ↑, house price ↓
• Larger |coefficient| → stronger effect on price
